#Importing Libraries

In [ ]:
!pip install flask flask-cors pyngrok ngrok PyPDF2 python-docx

In [ ]:
import markdown
from huggingface_hub import InferenceClient
from PIL import Image
from google.colab import userdata
import io
import torch
from transformers import AutoTokenizer, AutoModel, AutoProcessor

In [ ]:
from flask import Flask, send_file, jsonify, request
from flask_cors import CORS
from pyngrok import ngrok
from google.colab import userdata
import os
from huggingface_hub import InferenceClient
from openai import OpenAI

import PyPDF2
import docx
import io

#Chat API

In [ ]:
SYSTEM_MESSAGE_CONTENT = '''
an intelligent career assistant designed to help users improve their resumes, build strong portfolios, and prepare for job interviews.

You analyze resumes, suggest improvements, identify missing skills, and provide actionable feedback. You focus on clarity, impact, and industry relevance.

You also act as an interview coach:
- Ask relevant technical and HR questions
- Evaluate answers and give constructive feedback
- Suggest better ways to respond using structured frameworks

You generate professional portfolio content including:
- About Me sections
- Project descriptions
- Skill summaries

Guidelines:
- Be concise, clear, and practical
- Give specific suggestions (use numbers, metrics, action verbs)
- Avoid generic advice
- Adapt responses based on user's role (e.g., Software Engineer, Data Scientist)
- Be slightly conversational but professional

Always prioritize helping the user improve their career outcomes.

Tone:
- Friendly, confident, and slightly motivating
- Avoid robotic responses
- Use simple language

Behavior:
- If resume content is weak → give honest but helpful feedback
- If user asks for interview prep → switch to mock interview mode
- If user shares achievements → suggest stronger phrasing
- Never mention sharing the resume, since it'll be requested and shared earlier only
'''

def chat(query, resume):
    client = InferenceClient(
        model="moonshotai/Kimi-K2-Instruct-0905",
        token=userdata.get("HF_TOKEN"),
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_MESSAGE_CONTENT + "\n\n This is my resume" + resume
        },
        {
            "role": "user",
            "content": query
        }
    ]

    try:
        # Removed max_tokens to allow for longer responses without explicit limit
        completion = client.chat_completion(messages=messages, max_tokens=None)
        # --- Debugging prints ---
        print(f"DEBUG: Full completion object: {completion}")
        if completion.choices:
            print(f"DEBUG: Completion choices[0]: {completion.choices[0]}")
            print(f"DEBUG: Message content: {completion.choices[0].message.content}")
        else:
            print("DEBUG: completion.choices is empty or None.")
        # --- End Debugging prints ---

        if completion.choices and completion.choices[0].message.content is not None:
            return (completion.choices[0].message.content)
        else:
            print("DEBUG: AI model returned no content or empty content.")
            return ""
    except Exception as e:
        import traceback
        print(f"Error during chat_completion: {e}")
        traceback.print_exc()
        raise # Re-raise the exception after logging to ensure it's caught upstream

#UI Code

In [ ]:
html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Vantage AI | Portfolio Architect</title>
    <link href="https://fonts.googleapis.com/css2?family=Outfit:wght@300;400;700&display=swap" rel="stylesheet">
    <style>
        html, body {
            height: 100%;
            margin: 0;
            padding: 0;
        }

        :root {
            --bg: #050505;
            --panel: rgba(20, 20, 20, 0.6);
            --accent: #00f2ff;
            --accent-glow: rgba(0, 242, 255, 0.3);
            --text: #e0e0e0;
        }

        * { margin: 0; padding: 0; box-sizing: border-box; font-family: 'Outfit', sans-serif; }

        body {
            background: var(--bg);
            color: var(--text);
            /* height: 100vh; */ /* Managed by html, body height: 100% now */
            overflow: hidden;
            display: flex;
        }

        /* Left Panel: The Input Lab */
        .lab-side {
            width: 350px; /* Reduced width to give more space to the right */
            background: var(--panel);
            border-right: 1px solid rgba(255,255,255,0.1);
            backdrop-filter: blur(30px);
            display: flex;
            flex-direction: column;
            padding: 40px;
            z-index: 10;
            height: 100%; /* Ensure lab-side takes full height of body */
        }

        /* Right Panel: The Portfolio Preview */
        .preview-side {
            flex-grow: 1;
            background: linear-gradient(45deg, #080808 25%, #0f0f0f 100%);
            display: flex;
            align-items: center;
            justify-content: center;
            position: relative;
            padding: 50px;
            overflow-y: auto;
        }

        .preview-side::before {
            content: '';
            position: absolute;
            width: 300px; height: 300px;
            background: var(--accent);
            filter: blur(150px);
            opacity: 0.1;
            top: 10%; right: 10%;
        }

        /* UI Elements */
        .upload-circle {
            width: 80px; height: 80px;
            border: 2px solid var(--accent);
            border-radius: 50%;
            display: flex; align-items: center; justify-content: center;
            cursor: pointer;
            transition: 0.3s;
            box-shadow: 0 0 15px var(--accent-glow);
            margin-bottom: 20px;
        }

        .upload-circle:hover { transform: scale(1.1); background: var(--accent-glow); }

        .chat-area {
            flex-grow: 1;
            margin-top: 30px;
            display: flex;
            flex-direction: column;
        }

        .messages {
            flex-grow: 1;
            font-size: 0.9rem;
            color: #aaa;
            overflow-y: auto;
            height: 0; /* Ensures the flex item respects its container's height and scrolls */
        }

        /* Custom Scrollbar Styles for Webkit browsers */
        .messages::-webkit-scrollbar,
        .portfolio-mockup::-webkit-scrollbar,
        .input-box::-webkit-scrollbar {
            width: 8px;
        }

        .messages::-webkit-scrollbar-track,
        .portfolio-mockup::-webkit-scrollbar-track,
        .input-box::-webkit-scrollbar-track {
            background: transparent;
        }

        .messages::-webkit-scrollbar-thumb,
        .portfolio-mockup::-webkit-scrollbar-thumb,
        .input-box::-webkit-scrollbar-thumb {
            background-color: rgba(0, 242, 255, 0.3);
            border-radius: 10px;
            border: 2px solid var(--panel);
        }

        .messages::-webkit-scrollbar-thumb:hover,
        .portfolio-mockup::-webkit-scrollbar-thumb:hover,
        .input-box::-webkit-scrollbar-thumb:hover {
            background-color: var(--accent);
        }

        .bot-txt {
             color: var(--accent);
             margin-bottom: 20px;
             line-height: 1.6;
             word-wrap: break-word;
             white-space: pre-wrap;
        }

        /* Markdown specific styles for bot messages */
        .bot-txt ul, .bot-txt ol {
            margin-left: 20px;
            margin-top: 10px;
            margin-bottom: 10px;
            list-style-type: disc;
        }

        .bot-txt ol {
            list-style-type: decimal;
        }

        .bot-txt li {
            margin-bottom: 5px;
            line-height: 1.4;
            color: var(--text);
        }

        .bot-txt strong {
            color: var(--accent);
            font-weight: 700;
        }

        .chat-input-container {
            display: flex;
            gap: 10px;
            align-items: flex-end;
            width: 100%;
        }

        .input-box {
            background: rgba(255,255,255,0.05);
            border: 1px solid rgba(255,255,255,0.1);
            border-radius: 12px;
            padding: 15px;
            color: white;
            outline: none;
            flex-grow: 1;
            min-height: 40px;
            resize: vertical;
            max-height: 150px; /* Limit textarea height */
        }

        .send-button {
            background: var(--accent);
            color: var(--bg);
            border: none;
            border-radius: 12px;
            padding: 10px 20px;
            cursor: pointer;
            font-weight: 700;
            transition: background 0.3s ease, transform 0.1s ease;
            min-width: 80px;
            height: 40px; /* Match initial input box height */
            display: flex;
            align-items: center;
            justify-content: center;
        }

        .send-button:hover {
            background: var(--text);
        }

        .send-button.stop-state {
            background: #e74c3c; /* Red color for stop button */
        }

        .send-button.stop-state:hover {
            background: #c0392b;
        }

        /* Portfolio Components */
        .portfolio-mockup {
            width: 100%;
            max-width: 900px;
            background: #000;
            border: 1px solid rgba(255,255,255,0.1);
            border-radius: 20px;
            padding: 60px;
            opacity: 0.1;
            transform: translateY(20px);
            transition: 0.8s all cubic-bezier(0.16, 1, 0.3, 1);
            overflow-y: auto;
            max-height: 90vh;
        }

        .portfolio-mockup.active {
            opacity: 1;
            transform: translateY(0);
        }

        /* Portfolio Dynamic Content Styling */
        #portfolio-dynamic-content h2 {
            font-size: 2.2rem;
            font-weight: 700;
            color: var(--accent);
            margin-top: 40px;
            margin-bottom: 20px;
            border-bottom: 1px solid rgba(255,255,255,0.1);
            padding-bottom: 10px;
        }

        #portfolio-dynamic-content h3 {
            font-size: 1.5rem;
            font-weight: 600;
            color: var(--text);
            margin-top: 25px;
            margin-bottom: 15px;
        }

        #portfolio-dynamic-content p {
            font-size: 1rem;
            color: #ccc;
            line-height: 1.6;
            margin-bottom: 15px;
        }

        #portfolio-dynamic-content ul {
            list-style-type: disc;
            margin-left: 25px;
            margin-bottom: 15px;
        }

        #portfolio-dynamic-content ol {
            list-style-type: decimal;
            margin-left: 25px;
            margin-bottom: 15px;
        }

        #portfolio-dynamic-content li {
            font-size: 1rem;
            color: #ccc;
            line-height: 1.6;
            margin-bottom: 8px;
        }

        #portfolio-dynamic-content strong {
            color: var(--accent);
            font-weight: 700;
        }

        /* Decorative Pulse */
        .sync-pulse {
            display: inline-block;
            width: 8px; height: 8px;
            background: var(--accent);
            border-radius: 50%;
            margin-right: 10px;
            animation: pulse 1.5s infinite;
        }

        /* Styles for the details/summary element */
        details.suggestions-box {
            background: rgba(0, 242, 255, 0.05);
            border: 1px solid rgba(0, 242, 255, 0.2);
            border-radius: 8px;
            margin-bottom: 20px;
            padding: 10px 15px;
            color: var(--text);
            font-size: 0.9rem;
            line-height: 1.5;
        }

        details.suggestions-box summary {
            font-weight: 700;
            color: var(--accent);
            cursor: pointer;
            padding: 5px 0;
            outline: none;
        }

        details.suggestions-box summary::-webkit-details-marker {
            display: none !important;
        }

        details.suggestions-box summary::before {
            content: '▶';
            margin-right: 8px;
            transition: transform 0.2s ease-in-out;
            display: inline-block;
        }

        details.suggestions-box[open] summary::before {
            content: '▼';
            transform: rotate(90deg);
        }

        details.suggestions-box .suggestions-content {
            padding-top: 10px;
            border-top: 1px solid rgba(0, 242, 255, 0.1);
            margin-top: 10px;
        }

        details.suggestions-box .suggestions-content p,
        details.suggestions-box .suggestions-content ul,
        details.suggestions-box .suggestions-content ol {
            margin-bottom: 10px;
            color: #ccc;
        }

        details.suggestions-box .suggestions-content li {
            color: #ccc;
        }

        @keyframes pulse {
            0% { transform: scale(1); box-shadow: 0 0 0 0 rgba(0, 242, 255, 0.7); }
            70% { transform: scale(1.2); box-shadow: 0 0 0 10px rgba(0, 242, 255, 0); }
            100% { transform: scale(1); box-shadow: 0 0 0 0 rgba(0, 242, 255, 0); }
        }
    </style>
</head>
<body>

    <div class="lab-side">
        <h1 style="font-weight: 700; letter-spacing: -1px; margin-bottom: 40px;">VANTAGE<span style="color:var(--accent)">.</span></h1>

        <input type="file" id="fileUpload" accept=".pdf,.docx" style="display: none;">
        <div class="upload-circle" id="uploadBtn">
            <svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4"></path><polyline points="17 8 12 3 7 8"></polyline><line x1="12" y1="3" x2="12" y2="15"></line></svg>
        </div>
        <p style="font-size: 0.8rem; letter-spacing: 1px; color: #666; margin-bottom: 40px;">UPLOAD RESUME TO INITIALIZE</p>

        <div class="chat-area">
            <div class="messages" id="msgBox">
                <div class="bot-txt">System Offline. Waiting for source data...</div>
            </div>
            <div class="chat-input-container">
                <textarea id="chatInp" class="input-box" placeholder="Ask AI to refine content..." rows="1"></textarea>
            </div>
        </div>
    </div>

    <div class="preview-side">
        <div class="portfolio-mockup" id="portfolio">
            <div id="portfolio-dynamic-content">
                <!-- AI generated content will go here -->
                <p style="color: #888; text-align: center; margin-top: 50px;">Upload a resume to generate portfolio content and improvements.</p>
            </div>
        </div>
    </div>

<script>
    const uploadBtn = document.getElementById('uploadBtn');
    const fileUpload = document.getElementById('fileUpload');
    const portfolio = document.getElementById('portfolio');
    const msgBox = document.getElementById('msgBox');
    const chatInp = document.getElementById('chatInp');
    // Removed sendBtn from here
    const portfolioDynamicContent = document.getElementById('portfolio-dynamic-content');

    let abortController = null; // For cancelling fetch requests

    function scrollToBottom() {
        msgBox.scrollTop = msgBox.scrollHeight;
    }

    uploadBtn.onclick = () => {
        fileUpload.click(); // Trigger the hidden file input
    };

    fileUpload.addEventListener('change', async (event) => {
        const file = event.target.files[0];
        if (!file) return;

        msgBox.innerHTML += `<div class="bot-txt"><span class="sync-pulse"></span> Uploading and analyzing document...</div>`;
        scrollToBottom();

        const formData = new FormData();
        formData.append('resume', file);

        try {
            const response = await fetch('/upload_resume', {
                method: 'POST',
                body: formData
            });

            if (response.ok) {
                const data = await response.json();
                console.log('Received portfolio_content:', data.portfolio_content);
                console.log('Received improvement_suggestions:', data.improvement_suggestions);
                msgBox.innerHTML += `<div class="bot-txt"><span class="sync-pulse"></span> Document processed. Extracted ${data.text.length} characters.</div>`;
                scrollToBottom();
                portfolio.classList.add('active');

                if (data.portfolio_content) {
                    portfolioDynamicContent.innerHTML = data.portfolio_content;
                    msgBox.innerHTML += `<div class="bot-txt"><span class="sync-pulse"></span> Portfolio content generated!</div>`;
                    scrollToBottom();
                } else {
                    msgBox.innerHTML += `<div class="bot-txt" style="color: orange;">Warning: No portfolio content received.</div>`;
                    scrollToBottom();
                }

                if (data.improvement_suggestions) {
                    msgBox.innerHTML += `
                        <details class="bot-txt suggestions-box">
                            <summary>Resume Improvement Suggestions <span class="sync-pulse"></span></summary>
                            <div class="suggestions-content">${data.improvement_suggestions}</div>
                        </details>
                    `;
                    scrollToBottom();
                } else {
                    msgBox.innerHTML += `<div class="bot-txt" style="color: orange;">Warning: No improvement suggestions received.</div>`;
                    scrollToBottom();
                }

                setTimeout(() => {
                    msgBox.innerHTML += `<div class="bot-txt">Portfolio content generation initiated based on your resume. How do they look?</div>`;
                    scrollToBottom();
                }, 1500);
            } else {
                const errorData = await response.json();
                msgBox.innerHTML += `<div class="bot-txt" style="color: red;">Error: ${errorData.error}</div>`;
                scrollToBottom();
            }
        } catch (error) {
            msgBox.innerHTML += `<div class="bot-txt" style="color: red;">Network error or server unreachable: ${error.message}</div>`;
            scrollToBottom();
        }
    });

    async function processChatInput() {
        const val = chatInp.value.trim();
        if (!val) return;

        chatInp.value = '';
        chatInp.style.height = 'auto'; // Reset height for new input
        chatInp.disabled = true;
        // Removed sendBtn related logic here

        msgBox.innerHTML += `<div style="margin-bottom: 20px; text-align: right;">${val}</div>`;
        scrollToBottom();
        const thinkingMsg = `<div class="bot-txt thinking-message"><span class="sync-pulse"></span> Thinking...</div>`;
        msgBox.innerHTML += thinkingMsg;
        scrollToBottom();

        abortController = new AbortController();
        const signal = abortController.signal;

        try {
            const response = await fetch('/chat', {
                method: 'POST',
                headers: {
                    'Content-Type': 'application/json'
                },
                body: JSON.stringify({ query: val }),
                signal: signal
            });

            if (response.ok) {
                const data = await response.json();
                // Remove thinking message
                const lastThinkingMsg = msgBox.querySelector('.thinking-message');
                if (lastThinkingMsg) lastThinkingMsg.remove();

                msgBox.innerHTML += `<div class="bot-txt">${data.response}</div>`;
                scrollToBottom();

                // NEW: Update portfolio if portfolio_update is present
                if (data.portfolio_update) {
                    portfolioDynamicContent.innerHTML = data.portfolio_update;
                    msgBox.innerHTML += `<div class="bot-txt"><span class="sync-pulse"></span> Portfolio updated based on your request!</div>`;
                    scrollToBottom();
                }

            } else if (response.status === 400 && signal.aborted) {
                // This case handles a successful abortion where the server might still respond 400 due to cancelled request
                console.log('Request was aborted by the user.');
                const lastThinkingMsg = msgBox.querySelector('.thinking-message');
                if (lastThinkingMsg) lastThinkingMsg.remove();
                 msgBox.innerHTML += `<div class="bot-txt" style="color: orange;">AI response stopped.</div>`;
                 scrollToBottom();
            } else {
                const errorData = await response.json();
                // Remove thinking message
                const lastThinkingMsg = msgBox.querySelector('.thinking-message');
                if (lastThinkingMsg) lastThinkingMsg.remove();
                msgBox.innerHTML += `<div class="bot-txt" style="color: red;">Error: ${errorData.error}</div>`;
                scrollToBottom();
            }
        } catch (error) {
            // Remove thinking message
            const lastThinkingMsg = msgBox.querySelector('.thinking-message');
            if (lastThinkingMsg) lastThinkingMsg.remove();

            if (error.name === 'AbortError') {
                console.log('Fetch aborted');
                msgBox.innerHTML += `<div class="bot-txt" style="color: orange;">AI response stopped.</div>`;
            } else {
                msgBox.innerHTML += `<div class="bot-txt" style="color: red;">Network error or server unreachable: ${error.message}</div>`;
            }
            scrollToBottom();
        } finally {
            chatInp.disabled = false;
            // Removed sendBtn related logic here
            chatInp.focus();
            abortController = null;
        }

        portfolio.style.transform = "scale(0.98)";
        portfolio.style.borderColor = "var(--accent)";

        setTimeout(() => {
            portfolio.style.transform = "scale(1)";
            portfolio.style.borderColor = "rgba(255,255,255,0.1)";
        }, 600);
    }

    chatInp.addEventListener('keydown', (e) => {
        if (e.key === 'Enter' && !e.shiftKey) {
            e.preventDefault(); // Prevent default newline in textarea
            processChatInput();
        }
    });

    // Removed sendBtn click listener

    // Auto-resize textarea based on content
    chatInp.addEventListener('input', () => {
        chatInp.style.height = 'auto';
        chatInp.style.height = (chatInp.scrollHeight) + 'px';
    });

</script>
</body>
</html>
"""

with open("main.html", "w", encoding='utf-8') as f:
    f.write(html_content)

#App

In [ ]:
app = Flask(__name__)
CORS(app)

ngrok.kill()

ngrok.set_auth_token(userdata.get("NGROK_AUTH_TOKEN"))

PORT = 5000
public_url = ngrok.connect(PORT)
print(f' * ngrok tunnel available at: {public_url}')
print(f'You can access your HTML page at: {public_url}')

# Global variable to store resume content
resume_content = ""

@app.route('/')
def main():
    return send_file('main.html')

@app.route('/upload_resume', methods=['POST'])
def upload_resume():
    global resume_content # Declare intent to modify the global variable

    if 'resume' not in request.files:
        return jsonify({"error": "No file part"}), 400

    file = request.files['resume']
    if file.filename == '':
        return jsonify({"error": "No selected file"}), 400

    if file:
        filename = file.filename
        file_extension = filename.split('.')[-1].lower()
        extracted_text = ""

        if file_extension == 'pdf':
            try:
                reader = PyPDF2.PdfReader(io.BytesIO(file.read()))
                for page_num in range(len(reader.pages)):
                    extracted_text += reader.pages[page_num].extract_text() or ""
            except Exception as e:
                return jsonify({"error": f"Error reading PDF: {str(e)}"}), 500
        elif file_extension == 'docx':
            try:
                doc = docx.Document(io.BytesIO(file.read()))
                for paragraph in doc.paragraphs:
                    extracted_text += paragraph.text + '\n'
            except Exception as e:
                return jsonify({"error": f"Error reading DOCX: {str(e)}"}), 500
        else:
            return jsonify({"error": "Unsupported file type. Only PDF and DOCX are supported."}), 400

        resume_content = extracted_text # Store extracted text globally

        # Call chat function to generate portfolio content
        portfolio_query = "Based on this resume, generate a professional portfolio section. For the main sections (About Me, Project Descriptions, Skill Summaries), use H2 markdown headings (e.g., '## About Me'). For individual projects and skills, use H3 markdown headings (e.g., '### Project Title'). Ensure the content uses markdown formatting directly, with a newline after each heading, and without including markdown block delimiters like '```markdown' or '```'. Do not include any introductory or concluding remarks outside the markdown content. Also, skills' summaries shouldn't be too long."
        # Call chat function to generate resume improvement suggestions
        suggestions_query = "Based on this resume, provide 3 key suggestions for resume improvement. Generate the content using markdown formatting directly, without including markdown block delimiters like '```markdown' or '```'. Do not include any introductory or concluding remarks outside the markdown content."

        try:
            ai_portfolio_markdown = chat(portfolio_query, resume_content)
            print(f"Raw AI Portfolio Markdown: {ai_portfolio_markdown}") # Debugging line
            ai_portfolio_html = markdown.markdown(ai_portfolio_markdown)
            print(f"Generated AI Portfolio HTML: {ai_portfolio_html}") # New debugging line

            ai_suggestions_markdown = chat(suggestions_query, resume_content)
            ai_suggestions_html = markdown.markdown(ai_suggestions_markdown)

            return jsonify({"message": "File processed successfully", "text": extracted_text, "portfolio_content": ai_portfolio_html, "improvement_suggestions": ai_suggestions_html}), 200
        except Exception as e:
            return jsonify({"error": f"Error generating AI content: {str(e)}"}), 500

    return jsonify({"error": "Something went wrong during upload"}), 500

@app.route('/chat', methods=['POST'])
def chat_endpoint():
    user_query = request.json.get('query').lower() # Convert to lowercase for keyword matching
    if not user_query:
        return jsonify({"error": "No query provided"}), 400

    chat_response_text = ""
    portfolio_update_html = None # Initialize to None

    # Check for portfolio editing intent
    portfolio_keywords = ["edit portfolio", "update portfolio", "change portfolio", "refine portfolio", "modify portfolio", "add skill", "add to portfolio", "include skill", "update skill", "add new skill"]
    is_portfolio_edit_request = any(keyword in user_query for keyword in portfolio_keywords)

    # Check for resume editing intent
    resume_keywords = ["edit resume", "update resume", "change resume", "refine resume", "modify resume"]
    is_resume_edit_request = any(keyword in user_query for keyword in resume_keywords)

    try:
        if is_portfolio_edit_request:
            # Instruct AI to edit the portfolio based on the query and resume content
            portfolio_edit_prompt = f"Based on the following request: '{user_query}', please regenerate the complete professional portfolio content from the provided resume. For the main sections (About Me, Project Descriptions, Skill Summaries), use H2 markdown headings. For individual projects and skills, use H3 markdown headings. Ensure the content uses markdown formatting directly, with a newline after each heading, and without including markdown block delimiters like '```markdown' or '```'. Do not include any introductory or concluding remarks outside the markdown content."
            ai_portfolio_markdown = chat(portfolio_edit_prompt, resume_content)
            print(f"DEBUG (chat_endpoint): Raw AI Portfolio Markdown for update: {ai_portfolio_markdown}") # Debugging line
            portfolio_update_html = markdown.markdown(ai_portfolio_markdown)
            print(f"DEBUG (chat_endpoint): Generated AI Portfolio HTML for update: {portfolio_update_html}") # Debugging line
            chat_response_text = "I have updated the portfolio content on the right based on your request. Please review it."

        elif is_resume_edit_request:
            # Instruct AI to provide suggestions for resume improvement
            suggestions_prompt = f"Based on the user's request '{user_query}', provide 3 key suggestions for resume improvement. Also, clearly state that direct editing of the resume file is not possible. Generate the content using markdown formatting directly, without including markdown block delimiters like '```markdown' or '```'. Do not include any introductory or concluding remarks outside the markdown content."
            ai_suggestions_markdown = chat(suggestions_prompt, resume_content)
            chat_response_text = markdown.markdown(ai_suggestions_markdown)
            chat_response_text += "<br><br>Please note: I cannot directly edit your resume file, but I can only provide suggestions."

        else:
            # Regular chat
            response_text = chat(user_query, resume_content)
            chat_response_text = markdown.markdown(response_text)

        # Return both chat response and optional portfolio update
        return jsonify({"response": chat_response_text, "portfolio_update": portfolio_update_html}), 200

    except Exception as e:
        return jsonify({"error": f"Error processing chat: {str(e)}"}), 500

app.run(port=PORT)

 * ngrok tunnel available at: NgrokTunnel: "https://cc2b-35-227-70-172.ngrok-free.app" -> "http://localhost:5000"
You can access your HTML page at: NgrokTunnel: "https://cc2b-35-227-70-172.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 18:10:15] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 18:10:16] "GET /favicon.ico HTTP/1.1" 404 -


DEBUG: Full completion object: ChatCompletionOutput(choices=[ChatCompletionOutputComplete(finish_reason='stop', index=0, message=ChatCompletionOutputMessage(role='assistant', content='## About Me\nFinal-year B.Tech (CSE) student who turns Python & ML curiosity into working products.  \nI love shrinking the gap between a notebook experiment and a tool people actually use—whether that’s an AI chatbot that answers 85 % of FAQs without human help or a resume analyzer that cuts screening time by 40 %.  \nWhen I’m not tuning transformers you’ll find me automating my own workflow: Git hooks, SQL dashboards, or a quick Bash one-liner that saves the team an hour a week.\n\n## Projects\n\n### AI Chatbot (Python, NLTK, HuggingFace, Flask)\nEnd-to-end conversational agent for 5 000+ monthly student inquiries.  \n- Fine-tuned DistilBERT on 200 k Q&A pairs → 92 % intent accuracy.  \n- Added context-aware memory with Redis cache; 1.8 s avg response time.  \n- Containerized with Docker, deployed on AW

INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 18:11:16] "POST /upload_resume HTTP/1.1" 200 -


DEBUG: Full completion object: ChatCompletionOutput(choices=[ChatCompletionOutputComplete(finish_reason='stop', index=0, message=ChatCompletionOutputMessage(role='assistant', content=' **1. Quantify Technical Depth in Project Descriptions**\nYour current project bullets ("Built a chatbot using NLP techniques") miss the specifics hiring managers scan for. Replace vague statements with architecture details and metrics:\n\n- **Instead of:** *"Built a chatbot using NLP techniques"*\n- **Write:** *"Developed intent-classification chatbot using Python, spaCy, and scikit-learn; achieved 87% accuracy on custom dataset of 500+ conversational queries, reducing response latency to <200ms via caching layer"*\n\n- **Instead of:** *"Developed a tool to analyze resumes"*\n- **Write:** *"Engineered NLP-based resume parser using Python and regular expressions; processed 50+ file formats (PDF/DOCX), extracting key skills with 92% precision and exposing REST API via Flask"*\n\n**2. Elevate Your Professio

INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 18:12:51] "POST /chat HTTP/1.1" 200 -


DEBUG: Full completion object: ChatCompletionOutput(choices=[ChatCompletionOutputComplete(finish_reason='stop', index=0, message=ChatCompletionOutputMessage(role='assistant', content=' ## About Me\n\nResults-driven Computer Science student specializing in Artificial Intelligence and Machine Learning, with a proven track record of developing practical applications using Python. Currently pursuing B.Tech (2024–2028) while building intelligent systems that solve real-world problems. A collaborative team player who thrives in agile environments, contributing technical expertise while supporting peers to achieve collective goals. Passionate about writing clean, scalable code and continuously expanding knowledge in emerging technologies.\n\n## Project Descriptions\n\n### AI Chatbot\n\nArchitected and deployed an intelligent conversational agent using Natural Language Processing techniques including tokenization, intent classification, and named entity recognition. Implemented dialogue manage

INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 18:14:08] "POST /chat HTTP/1.1" 200 -


DEBUG: Full completion object: ChatCompletionOutput(choices=[ChatCompletionOutputComplete(finish_reason='stop', index=0, message=ChatCompletionOutputMessage(role='assistant', content='Quick ATS reality check:  \nScore ≈ 25–30 / 100.  \nThe parser is hitting blanks in almost every field it weights heaviest.\n\nWhere the bots are giving you zero points\n1. Zero measurable impact – no numbers, %, users, latency, accuracy, repo stars, downloads.  \n2. Generic skill list – “Python” and “Machine Learning” appear with no context, tools, or versions.  \n3. No keywords for the role you want – missing: TensorFlow, PyTorch, REST API, Flask, Docker, AWS, Agile, CI/CD, pandas, scikit-learn, etc.  \n4. No experience section – internships, freelance, OSS contributions, TA/RA work all count.  \n5. No education details – university name, city, GPA/percentage, relevant coursework (Data Structures, Algorithms, Linear Algebra).  \n6. File name & format – if you save it as “Resume_final(3).pdf” or use fanc